# [재실행 전용 · 2026-10-04 회계 수정] kospi_hrl_generalization_colab.ipynb

원본 `kospi_hrl_generalization_colab.ipynb`에서 **풀 학습 셀(§6)을 빼고, 저장된 4β 풀(models/kospi_hrl_pool/)로 평가만 다시 하는 노트북이다**. 환경은 회계 수정판(`_drift_weights` 등)이 들어간 원본 셀 그대로다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과는 원본과 같은 이름으로 저장되고, 실행 전 백업 셀이 수정 전 결과를 `_pre_accounting_fix_20261004/`로 먼저 복사한다(이미 복사된 파일은 건너뜀).

결과: 7장 KOSPI HRL 절(α 스윕 valid, 최종 valid/test 표).


In [ ]:
# 수정 전 결과 백업 — 파일 단위로, 이미 백업된 파일은 덮어쓰지 않음
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data'
dst = os.path.join(src, '_pre_accounting_fix_20261004')
os.makedirs(dst, exist_ok=True)
copied = []
for pat in ['kospi_hrl_*.csv']:
    for p in glob.glob(os.path.join(src, pat)):
        target = os.path.join(dst, os.path.basename(p))
        if not os.path.exists(target):
            shutil.copy2(p, target)
            copied.append(os.path.basename(p))
print('백업 폴더:', dst)
print('이번에 백업한 파일:', copied if copied else '없음(이미 백업됨)')


# KOSPI HRL 일반화 검증 (`paper/DRAFT.md`/`DRAFT_KR.md` 7장 대응)

크립토 트랙(`crypto/notebooks/crypto_hrl_earnhft_colab.ipynb`)에서 나온 최종 배포 레시피 —
**Q-teacher 없는 β-조건부 전문가 풀(Stage A/B) + 라우터 없이 추론 시점 관성 스무딩만 적용** — 이
독립적으로 구축된 두 번째 시장(KOSPI)에도 통하는지 확인한다.

**범위(중요, 반드시 먼저 읽을 것)**: 이 노트북은 크립토 트랙의 v1→v2→v3→v4→라우터붕괴 **전체
캐스케이드를 재현하지 않는다** — 그건 7장에서 이미 "4장 대부분을 두 번째 데이터셋에서 반복하는
것에 맞먹는 상당한 작업"이라 범위 밖으로 명시한 바 있다. 대신 **이미 검증이 끝난 최종 레시피만
이식**해 더 좁고 저비용인 질문에 답한다:

1. **다양성**: Q-teacher 없이 β-조건부 우선순위 샘플링만으로 학습한 KOSPI 전문가 풀도 v3처럼
   근균등가중으로 수렴하는가, 아니면 다양성이 유지되는가? (KOSPI는 5클러스터뿐이라 44종목에서
   봤던 붕괴가 재현될지 자체가 불확실 — 행동공간이 훨씬 작아 애초에 붕괴할 자유도가 적을 수 있다.)
2. **관성 스무딩**: 크립토에서 유일하게 확실히 통했던 개입(추론 시점 α-스무딩)이 여기서도 등가중과
   기존 단일 PPO 베이스라인(`rl_trading_colab.ipynb`, test +0.28%/Sharpe 0.100)을 이기게 해주는가?
3. **valid/test 일관성**: §4.6/§16과 동일한 잣대로, 여기서 "최선"으로 보이는 β/α 조합이 진짜
   일관된 우위인지 확인한다.

top-K 희소선택+리스크게이트(v4)와 라우터(Stage C)는 **의도적으로 이식하지 않는다** — 전자는 44종목
행동공간의 밀집 softmax 붕괴에 대한 구조적 해법이라 5차원 행동공간에는 애초에 적용 동기가 약하고,
후자는 크립토에서 최종적으로 "기여 0"으로 확정된 구성요소이기 때문이다(§15). 1번 진단에서 만약
다양성 붕괴가 실제로 재현된다면, KOSPI용 구조적 해법 설계는 이 노트북 결과를 보고 별도로 판단한다.

**데이터 분할**: 기존 단일 PPO 베이스라인은 train 2021–2023/test 2024–2025였다. 이 노트북은 Stage
A/B 학습에 **2023년을 아예 쓰지 않고**(train 2021–2022) 2023년을 진짜 미노출 valid로 확보한다 —
크립토 §1.3의 "클린 valid" 교훈(선택에 쓰인 구간과 최종 평가 구간을 분리)을 처음부터 설계에
반영한 것. test(2024–2025)는 기존 베이스라인과 완전히 동일해 직접 비교 가능하다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


## 1. 설정

In [ ]:
import warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
MODEL_DIR  = DRIVE_ROOT / 'models'

RL_EMB_H5   = DATA_DIR / 'rl_embeddings.h5'
PARQUET     = DATA_DIR / 'kospi_valid.parquet'
CLUSTER_CSV = DATA_DIR / 'cluster_assignments.csv'

EMB_DIM    = 64
N_CLUSTERS = 5
TC         = 0.001
LOOKBACK   = 60

# 3-way 분할 — 기존 rl_trading_colab.ipynb(train 2021-2023/test 2024-2025)와 달리 2023을
# 학습에서 완전히 제외해 진짜 미노출 valid로 확보한다(§1.3 클린 valid 교훈을 설계에 선반영).
TRAIN_START = '2021-01-01'
TRAIN_END   = '2022-12-31'
VALID_START = '2023-01-01'
VALID_END   = '2023-12-31'
TEST_START  = '2024-01-01'   # 기존 베이스라인과 동일 — 직접 비교 가능
TEST_END    = '2025-12-31'

N_LABELS       = 5
RISK_THRESHOLD = 0.2
LOWPASS_WINDOW = 20   # 크립토의 48(30분봉=1일)에 대응하는 일봉 스케일(~1개월 거래일)
POOL_BETAS         = [-90, -30, 30, 90]
POOL_TIMESTEPS_EACH = 100_000   # 크립토 §5·EIIE·LSRE-CAAN 노트북과 동일 예산(비교 가능하게)
SMOOTH_ALPHAS       = [1.0, 0.3, 0.15, 0.05, 0.03, 0.02, 0.01]

KOSPI_HRL_MODEL_DIR = MODEL_DIR / 'kospi_hrl_pool'
KOSPI_HRL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert RL_EMB_H5.exists(), f'rl_embeddings.h5 없음 — hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')


## 2. RL 임베딩 로드 & 인덱스 구축

In [ ]:
print('rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    tickers_raw = f['tickers'][:]
    dates_raw   = f['dates'][:]
    embs_all    = f['embeddings'][:]   # (N, 64)

tickers_list = [t.decode() for t in tickers_raw]
dates_list   = [d.decode() for d in dates_raw]

emb_index = {}
for i, (t, d) in enumerate(zip(tickers_list, dates_list)):
    emb_index[(t, d)] = i

print(f'임베딩: {embs_all.shape} | 인덱스: {len(emb_index):,}개')


## 3. 가격 데이터 & 클러스터 로드

In [ ]:
print('parquet 로드 중...')
raw_df = pd.read_parquet(PARQUET)
raw_df['Date'] = pd.to_datetime(raw_df['Date'])

cluster_df = pd.read_csv(CLUSTER_CSV)
cluster_df['종목코드'] = cluster_df['종목코드'].astype(str).str.zfill(6)
print(f'클러스터 분포:\n{cluster_df["cluster"].value_counts().sort_index()}')

cluster_tickers = defaultdict(list)
for _, row in cluster_df.iterrows():
    cluster_tickers[int(row['cluster'])].append(str(row['종목코드']))

raw_df['종목코드'] = raw_df['종목코드'].astype(str).str.zfill(6)
raw_df = raw_df.sort_values(['종목코드', 'Date'])

if 'RetTarget_1d' not in raw_df.columns:
    raw_df['RetTarget_1d'] = raw_df.groupby('종목코드')['Adj_Close'].pct_change()

ret_df = raw_df[['종목코드', 'Date', 'RetTarget_1d']].copy()
ret_df['date_str'] = ret_df['Date'].dt.strftime('%Y-%m-%d')

ret_index = {}
for _, row in ret_df.iterrows():
    if not np.isnan(row['RetTarget_1d']):
        ret_index[(str(row['종목코드']), row['date_str'])] = float(row['RetTarget_1d'])

all_date_strs = sorted(set(d for _, d in emb_index.keys()))
print(f'\n전체 날짜 수: {len(all_date_strs)}')


## 4. KospiPortfolioEnv (Gymnasium) — `rl_trading_colab.ipynb`와 동일, 변경 없음

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class KospiPortfolioEnv(gym.Env):
    '''
    KOSPI DTW 클러스터 기반 포트폴리오 운용 환경 (rl_trading_colab.ipynb §3과 동일)

    State:  5개 클러스터 평균 임베딩(5×64) + 현재 비중(5) → 325-dim
    Action: 목표 포트폴리오 비중 (5-dim, Softmax 정규화)
    Reward: 일일 포트폴리오 수익률 − 거래비용
    '''
    metadata = {'render_modes': []}

    def __init__(self, dates, emb_index, embs_all, ret_index,
                 cluster_tickers, n_clusters=5, emb_dim=64, tc=0.001):
        super().__init__()
        self.dates           = dates
        self.emb_index       = emb_index
        self.embs_all        = embs_all
        self.ret_index       = ret_index
        self.cluster_tickers = cluster_tickers
        self.n_clusters      = n_clusters
        self.emb_dim         = emb_dim
        self.tc              = tc

        obs_dim = n_clusters * emb_dim + n_clusters
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(n_clusters,), dtype=np.float32)

    def _cluster_emb(self, date_str):
        cluster_embs = []
        for c in range(self.n_clusters):
            embs_c = []
            for ticker in self.cluster_tickers[c]:
                idx = self.emb_index.get((ticker, date_str))
                if idx is not None:
                    embs_c.append(self.embs_all[idx])
            if embs_c:
                cluster_embs.append(np.mean(embs_c, axis=0))
            else:
                cluster_embs.append(np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(cluster_embs)

    def _cluster_ret(self, date_str):
        rets = []
        for c in range(self.n_clusters):
            rs = [self.ret_index.get((t, date_str), np.nan)
                  for t in self.cluster_tickers[c]]
            rs = [r for r in rs if not np.isnan(r)]
            rets.append(float(np.mean(rs)) if rs else 0.0)
        return np.array(rets, dtype=np.float32)

    def _get_obs(self):
        date_str = self.dates[self.t]
        cluster_e = self._cluster_emb(date_str)
        return np.concatenate([cluster_e, self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = np.ones(self.n_clusters, dtype=np.float32) / self.n_clusters
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action     = np.exp(action - action.max())
        action     = action / action.sum()

        turnover   = float(np.abs(action - self.weights).sum())
        tc_cost    = self.tc * turnover

        next_t     = min(self.t + 1, len(self.dates) - 1)
        next_date  = self.dates[next_t]
        cluster_rets = self._cluster_ret(next_date)
        port_ret   = float((action * cluster_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, cluster_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.dates) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover}
        return self._get_obs(), reward, done, False, info


In [ ]:
# 3-way 날짜 분할 + 환경 준비
train_dates = [d for d in all_date_strs if TRAIN_START <= d <= TRAIN_END]
valid_dates = [d for d in all_date_strs if VALID_START <= d <= VALID_END]
test_dates  = [d for d in all_date_strs if TEST_START  <= d <= TEST_END]
print(f'train: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)}일)')
print(f'valid: {valid_dates[0]} ~ {valid_dates[-1]} ({len(valid_dates)}일)')
print(f'test : {test_dates[0]} ~ {test_dates[-1]} ({len(test_dates)}일)')

env_kwargs = dict(
    emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, cluster_tickers=cluster_tickers,
    n_clusters=N_CLUSTERS, emb_dim=EMB_DIM, tc=TC,
)

print('\n환경 유효성 검사...')
check_env(KospiPortfolioEnv(train_dates[:100], **env_kwargs), warn=True)
print('OK')


## 5. Stage A — 시장 레짐 분할 (EarnHFT Algorithm 3, 크립토 §4와 동일 코드 재사용)

In [ ]:
# market_segmentation.py 이식 — 크립토 노트북 §4와 완전히 동일한 코드(인덱스 무관하게 동작하는
# 범용 함수라 버킷/날짜 구분 없이 그대로 재사용 가능)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
# 5클러스터 등가중 NAV 곡선 구성 (train_dates 기준, 거래비용 제외)
eq_ret_per_date = np.array([
    np.mean([ret_index.get((t, d), 0.0) for c in range(N_CLUSTERS) for t in cluster_tickers[c]])
    for d in train_dates
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_date), index=range(len(train_dates)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.1f}일)")
print(seg.chunks['regime'].value_counts())

fig, ax = plt.subplots(figsize=(14, 4))
colors = {'bear': 'firebrick', 'pullback': 'lightsalmon', 'sideways': 'lightgray', 'rally': 'lightgreen', 'bull': 'darkgreen'}
for _, row in seg.chunks.iterrows():
    ax.axvspan(row['start_idx'], row['end_idx'], color=colors[row['regime']], alpha=0.5)
ax.plot(eq_nav.values, color='black', lw=1)
ax.set_title('KOSPI 5클러스터 등가중 NAV + Stage A 레짐 분할 (2021-2022 학습 구간)')
ax.set_xlabel('거래일 인덱스 (train_dates 내 위치)')
plt.tight_layout()
plt.show()


## 6. Stage B — 레짐 선호도 기반 β풀 학습 (dense softmax, top-K/게이트 없음)

In [ ]:
class ChunkedKospiPortfolioEnv(KospiPortfolioEnv):
    '''크립토 §5의 ChunkedCryptoPortfolioEnv와 동일한 β-우선순위 청크 샘플링을
    KospiPortfolioEnv에 이식 — reset()마다 레짐 우선순위(β)로 구간을 골라 그 구간만 진행.'''

    def __init__(self, all_dates, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_dates = all_dates
        self.chunks    = chunks.reset_index(drop=True)
        self.beta      = beta
        self.priority  = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng       = rng or np.random.default_rng()
        super().__init__(dates=all_dates[:2], **env_kwargs)  # reset()에서 실제 구간으로 교체됨

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.dates = self.all_dates[int(row['start_idx']):int(row['end_idx']) + 1]
        if len(self.dates) < 2:
            self.dates = self.all_dates[max(0, int(row['start_idx']) - 1):int(row['end_idx']) + 2]
        return super().reset(seed=seed, options=options)


# 스모크 테스트
_test_env = ChunkedKospiPortfolioEnv(
    all_dates=train_dates, chunks=seg.chunks, beta=90, rng=np.random.default_rng(0), **env_kwargs
)
for _ in range(3):
    _test_env.reset()
    print(f'β=90 샘플 구간: {len(_test_env.dates)}일 ({_test_env.dates[0]} ~ {_test_env.dates[-1]})')


In [ ]:
# 재실행용 로더 (idempotent)
if 'pool_models' not in globals() or not pool_models:
    pool_models = {b: PPO.load(str(KOSPI_HRL_MODEL_DIR / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('풀 에이전트 로드 완료:', list(pool_models.keys()))


## 7. 다양성 진단 — v3(크립토)처럼 근균등가중으로 수렴했는가?

In [ ]:
def run_solo_raw(model, dates):
    '''관성 스무딩 없이 원 정책 그대로 실행 — 다양성 진단용(비중 궤적 수집).'''
    env = KospiPortfolioEnv(dates, **env_kwargs)
    obs, _ = env.reset()
    done = False
    weight_history = [env.weights.copy()]
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
        weight_history.append(env.weights.copy())
    return np.array(env.value_history), np.array(weight_history), list(env.turnover_history)


print('=== [§7] β별 valid 구간 비중 궤적으로 다양성 진단 ===\n')
weight_traj = {}
for beta in POOL_BETAS:
    _, w_hist, turns = run_solo_raw(pool_models[beta], valid_dates)
    weight_traj[beta] = w_hist
    print(f'β={beta:>4d}: 클러스터별 평균 비중 = {w_hist.mean(axis=0).round(3)}, '
          f'평균 턴오버/스텝 = {np.mean(turns)*100:.2f}%')

print('\n--- β쌍별 비중 궤적 평균 상관(클러스터별 시계열 상관의 평균) ---')
betas = POOL_BETAS
corr_rows = {}
for b1 in betas:
    row = {}
    for b2 in betas:
        corrs = [np.corrcoef(weight_traj[b1][:, c], weight_traj[b2][:, c])[0, 1] for c in range(N_CLUSTERS)]
        row[f'β={b2}'] = np.nanmean(corrs)
    corr_rows[f'β={b1}'] = row
corr_df = pd.DataFrame(corr_rows).T
print(corr_df.to_string(float_format=lambda x: f'{x:.3f}'))
print('\n(대각선 제외 상관이 전부 ~1에 가까우면 v3와 같은 근균등가중 수렴 의심 — §7-B 판정 기준 참고)')


### 7-B. 판정 기준

1. 4개 β의 클러스터별 평균 비중(위 출력)이 서로 뚜렷이 다르면(예: 특정 β는 특정 클러스터에
   확연히 더 쏠림) 다양성이 있다고 본다 — v3의 "네 β가 사실상 클론"과 다른 결과.
2. 비중 궤적 간 상관이 대각선을 제외하고도 대부분 0.9 이상이면 v3와 유사한 근균등가중 수렴으로
   판정 — 이 경우 §8~9의 스무딩·비교는 그대로 진행하되, 논문에는 "5차원 행동공간에서도 Q-teacher
   없는 다양성 붕괴가 재현된다"는 결과로 보고한다(이 자체가 크립토 발견의 일반화 증거가 됨).
3. 다양성이 있는 것으로 판정되면, "44종목 규모에서만 나타나는 문제가 아니다"는 쪽으로 해석이
   갈리므로 §7의 결론(2026-07-18 v3 진단 시 "45차원 연속 softmax에서 다양성 부재"라고 특정했던
   차원 의존성 주장)을 재검토해야 한다는 점을 논문에 명시할 것.

## 8. 추론 시점 관성 스무딩 — β별 α 스윕 (valid에서 선택, log-trick으로 재학습 없이 적용)

In [ ]:
def run_solo_smoothed(model, alpha, dates):
    '''크립토 §9 run_hrl_smoothed의 KOSPI 이식 — KospiPortfolioEnv.step()이 내부에서
    softmax(action)을 적용하므로, softmax(log(w_new)) = w_new를 이용해 재학습 없이
    w_new=(1-α)w_old+α·softmax(raw_action)를 그대로 반영한다.'''
    env = KospiPortfolioEnv(dates, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        raw = np.exp(a - a.max())
        raw = raw / raw.sum()
        w = (1 - alpha) * env.weights + alpha * raw
        obs, _, done, _, info = env.step(np.log(np.maximum(w, 1e-12)))
    return np.array(env.value_history), list(env.turnover_history)


def perf_row_kospi(values, turnovers, tc=TC, steps_per_year=252):
    rets   = np.diff(values) / values[:-1]
    ann    = values[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((values / np.maximum.accumulate(values)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률': f'{(values[-1]-1)*100:.2f}%',
        '연환산':    f'{ann*100:.2f}%',
        'Sharpe':   f'{sharpe:.3f}',
        'MDD':      f'{mdd*100:.2f}%',
        '평균 턴오버/스텝': f'{t.mean()*100:.2f}%',
    }


print('=== [§8] β×α 스윕 — valid(2023)에서 최선의 조합 탐색 ===\n')
sweep_rows = {}
best_by_beta = {}
for beta in POOL_BETAS:
    best_val, best_alpha = -np.inf, None
    for alpha in SMOOTH_ALPHAS:
        v_vals, v_turns = run_solo_smoothed(pool_models[beta], alpha, valid_dates)
        row = perf_row_kospi(v_vals, v_turns)
        sweep_rows[f'β={beta}, α={alpha}'] = row
        sharpe_val = (np.diff(v_vals) / v_vals[:-1]).mean() / ((np.diff(v_vals) / v_vals[:-1]).std() + 1e-8) * np.sqrt(252)
        if sharpe_val > best_val:
            best_val, best_alpha = sharpe_val, alpha
    best_by_beta[beta] = (best_alpha, best_val)
    print(f'β={beta}: valid 최선 α={best_alpha} (Sharpe={best_val:.3f})')

sweep_df = pd.DataFrame(sweep_rows).T
print()
print(sweep_df.to_string())
sweep_df.to_csv(DATA_DIR / 'kospi_hrl_alpha_sweep_valid.csv', encoding='utf-8-sig')
print('\n저장 완료: kospi_hrl_alpha_sweep_valid.csv')


## 9. 최종 비교 — valid에서 고른 β/α를 test(2024-2025)에서 1회 실행, 등가중·기존 단일 PPO와 비교

In [ ]:
# valid Sharpe가 가장 높은 β를 최종 후보로 선택(§4.6/§16과 동일 원칙 — 여러 β 중 라우터 없이
# "가장 valid에서 일관되게 좋은 것" 하나를 고른다) — best_by_beta는 §8에서 계산됨
FINAL_BETA  = max(best_by_beta, key=lambda b: best_by_beta[b][1])
FINAL_ALPHA = best_by_beta[FINAL_BETA][0]
print(f'최종 후보: β={FINAL_BETA}, α={FINAL_ALPHA} (valid Sharpe={best_by_beta[FINAL_BETA][1]:.3f})\n')


def equal_weight_backtest_kospi(dates):
    env = KospiPortfolioEnv(dates, **env_kwargs)
    obs, _ = env.reset()
    done = False
    action = np.zeros(N_CLUSTERS, dtype=np.float32)  # softmax(0)=균등
    while not done:
        obs, _, done, _, info = env.step(action)
    return np.array(env.value_history), list(env.turnover_history)


print('=== [§9] 최종 비교 — valid(2023) / test(2024-2025, "1회만" 실행) ===\n')

v_final_vals, v_final_turns = run_solo_smoothed(pool_models[FINAL_BETA], FINAL_ALPHA, valid_dates)
t_final_vals, t_final_turns = run_solo_smoothed(pool_models[FINAL_BETA], FINAL_ALPHA, test_dates)
ew_valid_vals, ew_valid_turns = equal_weight_backtest_kospi(valid_dates)
ew_test_vals,  ew_test_turns  = equal_weight_backtest_kospi(test_dates)

comparison_valid = {
    '등가중 벤치마크(이 노트북 재계산)':          perf_row_kospi(ew_valid_vals, ew_valid_turns),
    f'KOSPI HRL β={FINAL_BETA}, α={FINAL_ALPHA}': perf_row_kospi(v_final_vals, v_final_turns),
}
comparison_test = {
    '등가중 벤치마크(이 노트북 재계산)':          perf_row_kospi(ew_test_vals, ew_test_turns),
    f'KOSPI HRL β={FINAL_BETA}, α={FINAL_ALPHA}': perf_row_kospi(t_final_vals, t_final_turns),
    # 참고용 — rl_trading_colab.ipynb에서 이미 확인된 값(동일 test 구간 2024-2025), 재계산 아님
    '단일 정책 PPO(참고치, rl_trading_colab.ipynb)': {'총 수익률': '+0.28%', 'Sharpe': '0.100'},
    '등가중(참고치, rl_trading_colab.ipynb 원본)':    {'총 수익률': '+34.33%', 'Sharpe': '1.043'},
}

valid_df = pd.DataFrame(comparison_valid).T
test_df  = pd.DataFrame(comparison_test).T

print('--- valid(2023) ---')
print(valid_df.to_string())
print('\n--- test(2024-2025) ---')
print(test_df.to_string())

valid_df.to_csv(DATA_DIR / 'kospi_hrl_final_valid.csv', encoding='utf-8-sig')
test_df.to_csv(DATA_DIR / 'kospi_hrl_final_test.csv', encoding='utf-8-sig')
print('\n저장 완료: kospi_hrl_final_{valid,test}.csv')
print('\n(주의: 참고치 2행은 이 노트북에서 재계산한 게 아니라 rl_trading_colab.ipynb 결과를 그대로')
print(' 옮긴 것 — 같은 test 구간(2024-2025)·같은 클러스터·같은 TC라 비교 가능하지만 재현은 아님.')
print(' 원본 등가중 참고치(+34.33%)와 이 노트북이 재계산한 등가중이 다르면 파이프라인 정합성 문제 —')
print(' 그 경우 결론은 이 노트북 안에서 재계산된 두 행(등가중 vs KOSPI HRL)끼리의 비교로만 유효.)')


## 10. 판정 기준

1. **1차(핵심)**: test에서 KOSPI HRL이 이 노트북이 재계산한 등가중 벤치마크를 이기는지, 그리고
   valid Sharpe와 test Sharpe의 격차가 크지 않은지(§4.6/§16과 동일 진단) 확인한다. 격차가 크면
   (특히 valid는 나쁜데 test만 좋으면) β=30·모멘텀과 같은 "우연히 test에서만 좋음" 패턴으로
   기각한다.
2. **2차**: 원본 단일 정책 PPO(참고치, +0.28%/Sharpe 0.100)를 확실히 앞서는지 확인한다 — 이기지
   못하면 "계층 구조가 KOSPI에서는 도움이 안 된다"는 부정적이지만 그 자체로 보고 가치 있는 결과.
3. **§7의 다양성 진단과 함께 읽을 것**: 다양성이 애초에 붕괴했다면(§7-B) 여기서 나온 성과 차이가
   "레짐 조건부 학습의 진짜 효과"가 아니라 "우연히 하나의 β가 다른 셋보다 나은 초기화로 수렴"에
   불과할 수 있다 — 두 결과를 분리해서 해석할 것.
4. 결과는 `paper/DRAFT.md`/`DRAFT_KR.md` 7장("이 간극을 좁히기")에 반영 — 재현되면 핵심 주장을
   상당히 강화하는 긍정적 결과, 재현되지 않으면(예: 다양성은 애초에 안 붕괴했거나, HRL이 단일
   정책보다 못하거나) 이 실패 모드들이 시장 미시구조에 의존한다는 정보가 되는 발견으로 보고한다.